In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator

In [ ]:
# Helper Functions
def pca_func(data: pd.DataFrame, title_suffix: str = "") -> tuple[PCA, pd.DataFrame]:
    """
    Perform PCA on standardized data and plot explained variance.
    Returns the fitted PCA object and PCA scores DataFrame.
    """
    pca = PCA(n_components=data.shape[1])
    scores = pca.fit_transform(data)

    # Plot explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(pca.explained_variance_, marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()
    

    # Print explained variance summary
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)

    # Return PCA model and scores
    df_scores = pd.DataFrame(
        scores,
        columns=[f'PC{i}' for i in range(1, scores.shape[1] + 1)],
        index=data.index
    )
    return pca, df_scores

In [ ]:
def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,
           pcx: int = 1, pcy: int = 2, new_axes: bool = False,
           title: str | None = None) -> None:
    """
    Create a PCA biplot between selected principal components.
    """
    x_label = f"PC{pcx}"
    y_label = f"PC{pcy}"

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(df_scores[x_label], df_scores[y_label], color="b", alpha=0.7)

    # Label axes with explained variance (if desired)
    if new_axes:
        explvar = 100 * pca.explained_variance_ratio_
        ax.set_xlabel(f"{x_label} ({explvar[pcx-1]:.1f}% explained var.)", fontsize=10)
        ax.set_ylabel(f"{y_label} ({explvar[pcy-1]:.1f}% explained var.)", fontsize=10)
    else:
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)

    # Equal aspect for accurate scaling
    ax.set_aspect('equal', adjustable='datalim')

    # Create second axes for loadings
    ax2 = ax.twinx().twiny()
    font = {'color': 'g', 'weight': 'bold', 'size': 10}

    for col in df_loadings.columns:
        tipx = df_loadings.loc[x_label, col]
        tipy = df_loadings.loc[y_label, col]
        ax2.arrow(0, 0, tipx, tipy, color="r", alpha=0.5, length_includes_head=True)
        ax2.text(tipx * 1.05, tipy * 1.05, col, fontdict=font, ha="center", va="center")

    # Align axes centers
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    ax2.set_aspect('equal', adjustable='datalim')

    # Title
    if title:
        plt.title(title)
    plt.tight_layout()
  

In [ ]:
# -----------------------------
# PCA on Personality Test Data
# -----------------------------

# -----------------------------
# Main Analysis
# -----------------------------

# Load and prepare data
df = pd.read_csv("data/personalitydata.csv", index_col=0)
print("Original Data Preview:\n", df.head())

#df = df.iloc[:, 1:]  # Remove first (ID) column

# Standardize
dfs = pd.DataFrame(
    StandardScaler().fit_transform(df),
    columns=df.columns,
    index=df.index
)


In [ ]:
# PCA
pca, df_scores = pca_func(dfs, " (Personality Data)")
df_loadings = pd.DataFrame(
    pca.components_,
    columns=df.columns,
    index=df_scores.columns
)

# -----------------------------
# Biplots for selected components
# -----------------------------

biplot(df_scores, df_loadings, pca, pcx=1, pcy=2, new_axes=True,
       title="Personality Data PCA Biplot (PC1 vs PC2)")

#biplot(df_scores, df_loadings, pca, pcx=1, pcy=3, new_axes=True,
#       title="Personality Data PCA Biplot (PC1 vs PC3)")

#biplot(df_scores, df_loadings, pca, pcx=1, pcy=4, new_axes=True,
#       title="Personality Data PCA Biplot (PC1 vs PC4)")
# plt.show()

In [ ]:
def pca_varimax(df, k, normalize=True):
    """
    Perform PCA with Varimax rotation on any numeric DataFrame.
 
    Parameters
    ----------
    df : pd.DataFrame
        Numeric dataframe (features x samples)
    k : int
        Number of principal components to extract
    normalize : bool
        Whether to normalize during varimax rotation
 
    Returns
    -------
    pca_k : fitted PCA object
    load_rot_df : rotated loadings DataFrame
    rotation_matrix : rotation matrix
    summary : DataFrame of SS loadings, proportion var, cumulative var
    """
 
    # 1. PCA
    pca_k = PCA(n_components=k, svd_solver="full", random_state=0)
    pca_k.fit(df)
    eigvecs = pca_k.components_.T  # shape (n_features, k)
 
    loadings_df = pd.DataFrame(
        eigvecs,
        index=df.columns,
        columns=[f"PC{i+1}" for i in range(k)]
    )
    #print("\nOriginal PCA Loadings (before rotation):")
    #print(pretty(loadings_df))
 
    # 2. Varimax rotation
    rotator = Rotator(method="varimax", normalize=normalize)
    loadings_rot = rotator.fit_transform(eigvecs)
    rotation_matrix = rotator.rotation_
 
    # 3. Rotated loadings DataFrame
    load_rot_df = pd.DataFrame(
        loadings_rot,
        index=df.columns,
        columns=[f"RPC{i+1}" for i in range(k)]
    )
 
    # 4. Variance summaries
    sq = load_rot_df ** 2
    ss = sq.sum(axis=0)
    prop = ss / load_rot_df.shape[0]
    cum = prop.cumsum()
    summary = pd.DataFrame(
        [ss, prop, cum],
        index=["SS loadings", "Proportion Var", "Cumulative Var"]
    )
 

In [ ]:
# -----------------------------
# PCA with Varimax Rotation
# -----------------------------
k=5
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(dfs)

#loadings_df = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2", "PC3"])

loadings_df = pd.DataFrame(
        pca.components_.T,
        index=df.columns,
        columns=[f"PC{i+1}" for i in range(k)]
    )

print(loadings_df)


In [ ]:
rotator = Rotator(method="varimax")
loadings_rot = rotator.fit_transform(pca.components_.T)

R = rotator.rotation_

#load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["RPC1", "RPC2", "RPC3"])
load_rot_df = pd.DataFrame(
        loadings_rot,
        index=df.columns,
        columns=[f"RPC{i+1}" for i in range(k)]
    )

print(load_rot_df)

In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than this value (cut),  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)
print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))